In [ ]:
# Python modules imports
import datetime
import calendar
import os

import boto3
import pandas as pd
import numpy as np
from io import StringIO
from scipy.ndimage import gaussian_filter
from datetime import datetime

import sys
from pathlib import Path

# In Jupyter, Path.cwd() replaces Path(__file__)
# It gets the directory where your notebook is saved
notebook_dir = Path.cwd().resolve()

# Step up to the project root where 'dags' folder is visible
# Adjust the number of '.parent' calls based on your notebook's depth
project_root = notebook_dir.parent.parent.parent
print(project_root)

# Add the project root to sys.path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

#local modules imports
from dags.common.py.snowflake.snowflakedb import SnowFlakeDB
from dags.common.py.utils.verbose_log import verbose, log
from dags.common.py.utils.utility_functions import aws_conn_credentials, upload_to_aws
from dags.engagement_scoring.configs import load_config


In [ ]:
ENVIRONMENT = 'dev'

EVTSCORE_DATABASE = 'DEV'
EVTSCORE_SCHEMA = 'MPM_EXCHANGE'
SNOWFLAKE_WAREHOUSE = 'BSM_ETL'


EVTSCORE_PROB_DDL = f"""
CREATE OR REPLACE TABLE {EVTSCORE_DATABASE}.{EVTSCORE_SCHEMA}.EVTSCORE_PROB (
  EVTSCORE_CONVERSION_TYPE 	STRING,
  ACCOUNT_TIER STRING,
  BIN 	FLOAT,
  MINSCORE 	FLOAT,
  MAXSCORE 	FLOAT,
  CONVERTERS 	FLOAT,
  ACCOUNTS 	FLOAT,
  CONVERSION_RATE 	FLOAT,
  SMOOTH_CONV_RATE FLOAT
)
"""

# EVTSCORE_COMPUTE select query.
EVTSCORE_COMPUTE_SELECT_SQL = f"""
SELECT
  EVTSCORE_CONVERSION_TYPE,
  ACCOUNT_TIER,
  FLOOR(POS*500/TOTAL_CONV) AS BIN,
  MIN(SCORE) AS MINSCORE,
  MAX(SCORE) AS MAXSCORE,
  SUM(CONV_FLAG) AS CONVERTERS,
  COUNT(*) AS ACCOUNTS
FROM
(
  SELECT
    *,
    SUM(CONV_FLAG) OVER(PARTITION BY EVTSCORE_CONVERSION_TYPE, ACCOUNT_TIER ORDER BY SCORE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS POS,
    SUM(CONV_FLAG) OVER(PARTITION BY EVTSCORE_CONVERSION_TYPE, ACCOUNT_TIER) AS TOTAL_CONV
  FROM {EVTSCORE_DATABASE}.{EVTSCORE_SCHEMA}.EVTSCORE_COMPUTE
) T
GROUP BY 1,2,3
HAVING COUNT(*) > 10
ORDER BY 1,2
"""

# to copy the data from S3 into EVSCORE_PROB table
COPY_INTO_SQL = """
COPY INTO {DATABASE}.MPM_EXCHANGE.EVTSCORE_PROB
      FROM @BSM_ETL.MPM_{ENVIRONMENT}.LANDINGZONE_ALLDATA_STAGE/evtscore/evtscore_prob.csv
      FILE_FORMAT = (FORMAT_NAME = BSM_ETL.MPM_PRD.CSV)
      FORCE=TRUE
""".format(DATABASE=EVTSCORE_DATABASE, ENVIRONMENT=ENVIRONMENT.upper())

In [ ]:
def get_evtscore_compute_dataframe(**kwargs):

    # create the snowflake connection object.
    try:

        snowflake_connection_evt = SnowFlakeDB(
            user='user',
            password='pwd',
            account='tata.us-east-1',
            warehouse=SNOWFLAKE_WAREHOUSE,
            database=EVTSCORE_DATABASE,
            schema=EVTSCORE_SCHEMA)
    except Exception as e:
        log(f"####### Exception: {e}")
    
    # create EVTSCORE_PROB table
    create_prob_table_result = snowflake_connection_evt._execute(EVTSCORE_PROB_DDL)
    temp_df = pd.DataFrame.from_dict(create_prob_table_result)
    log(f'create_prob_table_result for sf {EVTSCORE_DATABASE}.{EVTSCORE_SCHEMA}.EVTSCORE_PROB')
    log(f"EVTSCORE_PROB data: \n{temp_df.head}" )


    # get the EVTSCORE_COMPUTE records
    result_set = snowflake_connection_evt.getManyRows(EVTSCORE_COMPUTE_SELECT_SQL, limit=None)

    # convert it in to a Pandas dataframe
    df = pd.DataFrame.from_dict(result_set)
    df['CONVERSION_RATE'] = (df['CONVERTERS'] / df['ACCOUNTS']).replace([np.inf, -np.inf], np.nan).fillna(0)
    log(f"Conversion rate: \n{df['CONVERSION_RATE']}")

    conversion_types = list(df['EVTSCORE_CONVERSION_TYPE'].unique())
    account_tiers = list(df['ACCOUNT_TIER'].unique())
    final_df_list = []
    df.sort_values(['EVTSCORE_CONVERSION_TYPE', 'ACCOUNT_TIER', 'BIN'], ascending=[True, True, True], inplace=True)

    # loop through each of the items
    for conv_type in conversion_types:
        # filter for that conversion
        for account_tier in account_tiers:
            conv_df = df[(df['EVTSCORE_CONVERSION_TYPE']==conv_type) & (df['ACCOUNT_TIER']==account_tier)].copy()

            # IF converted dataframe is empty then continue with next 
            if conv_df.empty:
               continue

            conv_df.reset_index(inplace=True, drop=True)

            smooth_density = pd.DataFrame(gaussian_filter(conv_df['CONVERSION_RATE'], sigma=10, order=0),
                                          columns=['SMOOTH_CONV_RATE'])

            result_df = pd.concat([conv_df, smooth_density], axis=1, ignore_index=False)

            log("Squared error term")
            print(sum(pow(
                result_df['CONVERSION_RATE'][1:len(result_df) - 1] -
                np.roll(result_df['SMOOTH_CONV_RATE'], 1)[1:len(result_df) - 1], 2
            )))

            # append to the final dataframe
            final_df_list.append(result_df)
    

    final_df = pd.concat(final_df_list, ignore_index=True)
    display(final_df)
    display(df)
    log(f'len fo df: {len(final_df)}')
    log('starting s3 upload')

    try:
      aws_access_key, aws_secret_key = aws_conn_credentials()
      session = boto3.Session(aws_access_key_id=aws_access_key, aws_secret_access_key=aws_secret_key)
      s3_client = session.client('s3')

      # Convert Final DataFrame to CSV and upload
      csv_buffer = StringIO()
      final_df.to_csv(csv_buffer, index=False)
      
      # upload the file to S3
      s3_client.put_object(
          Bucket=s3_bucket_, 
          Key=s3_evt_path_, 
          Body=csv_buffer.getvalue()
        )

      log(f'>>>>>>>>>>>> s3 upload done')
    except Exception as e:
      log(f"########### Error uploading file to S3: {e}")
      raise

    log(f'>>>>>>>>>>>> copy start')
    snowflake_connection_evt._execute(COPY_INTO_SQL)
    log(f'>>>>>>>>>>>> copy done')
    return None

In [ ]:
get_evtscore_compute_dataframe()

In [ ]:
from datetime import datetime, timedelta

_3_days_before_dt = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=3)

# print(datetime.now())

execution_date = datetime.now()
# Get first day of current month
current_month_start = execution_date.replace(day=1)
print(current_month_start)

In [ ]:
from datetime import datetime, timedelta
import calendar

execution_date = datetime.now()

def _get_last_month_dates(execution_date: datetime) -> str:
    """
    Get last date for the last month
    """
    # Get first day of current month
    current_month_start = execution_date.replace(day=1)
    
    # Get first day of last month
    if current_month_start.month == 1:
        last_month_start = current_month_start.replace(year=current_month_start.year - 1, month=12)
    else:
        last_month_start = current_month_start.replace(month=current_month_start.month - 1)
    
    # Get last day of last month
    last_day = calendar.monthrange(last_month_start.year, last_month_start.month)[1]
    last_month_end = last_month_start.replace(day=last_day)

    last_day_of_last_month = last_month_end.strftime('%Y-%m-%d')
    first_day_of_last_month = last_month_start.strftime('%Y-%m-%d')

    print("::::: Last day of last month(func -> _get_last_month_dates):", last_day_of_last_month)
    print("::::: first day of last month(func -> _get_last_month_dates):", first_day_of_last_month)

    return last_day_of_last_month

_get_last_month_dates(execution_date)